In [1]:
import pandas as pd
import numpy as np

# 1) Load data
file_path = "/home/ducvu0904/Downloads/xstk/injury+stat.csv"
df = pd.read_csv(file_path)

# 2) Standardize common missing markers
missing_markers = ["", " ", "NA", "N/A", "na", "null", "NULL", "None", "-"]
df = df.replace(missing_markers, np.nan)

# Optional: normalize column names
df.columns = [c.strip() for c in df.columns]

# 3) Basic data quality checks
rows_before = len(df)
dup_count = int(df.duplicated().sum())
df = df.drop_duplicates().copy()
rows_after = len(df)

print("=== DATA OVERVIEW ===")
print(f"Rows before duplicate removal: {rows_before:,}")
print(f"Duplicate rows removed: {dup_count:,}")
print(f"Rows after duplicate removal: {rows_after:,}")
print(f"Columns: {df.shape[1]:,}")

# 4) Missing-value report for every feature
missing_count = df.isna().sum()
missing_pct = (missing_count / len(df) * 100).round(2)

nan_report = pd.DataFrame({
    "feature": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "missing_count": missing_count.values,
    "missing_pct": missing_pct.values,
    "n_unique": df.nunique(dropna=True).values
}).sort_values(["missing_count", "missing_pct"], ascending=False)

print("\n=== NaN REPORT (ALL FEATURES) ===")
display(nan_report)

print("\n=== FEATURES WITH NaN > 0 ===")
display(nan_report[nan_report["missing_count"] > 0])

# 5) Drop rows with NaN in required physical columns
required_cols_requested = ["player_height_inches", "player_weight"]
col_lookup = {c.lower(): c for c in df.columns}
missing_required = [c for c in required_cols_requested if c not in col_lookup]
if missing_required:
    raise KeyError(f"Required columns not found: {missing_required}")
required_cols = [col_lookup[c] for c in required_cols_requested]

rows_before_required_drop = len(df)
df = df.dropna(subset=required_cols).copy()
rows_dropped_required = rows_before_required_drop - len(df)
print("\n=== REQUIRED FIELDS CLEANING ===")
print(f"Rows dropped (NaN in {required_cols}): {rows_dropped_required:,}")
print(f"Rows remaining: {len(df):,}")

# Keep cleaned dataframe for next steps and save back to source CSV
df_clean = df.copy()
df_clean.to_csv(file_path, index=False)
print(f"\nSaved cleaned data to: {file_path}")

=== DATA OVERVIEW ===
Rows before duplicate removal: 5,578
Duplicate rows removed: 0
Rows after duplicate removal: 5,578
Columns: 32

=== NaN REPORT (ALL FEATURES) ===


,feature,dtype,missing_count,missing_pct,n_unique
27,TEAM,str,4364,78.24,30
28,INJURED_ON,str,4364,78.24,756
29,RETURNED,str,4364,78.24,736
30,DAYS_MISSED,float64,4364,78.24,64
31,INJURED_TYPE,str,4364,78.24,5
5,PLAYER_HEIGHT_INCHES,float64,6,0.11,22
6,PLAYER_WEIGHT,float64,6,0.11,124
0,PLAYER_ID,int64,0,0.00,1391
1,PLAYER_NAME,str,0,0.00,1389
2,SEASON,str,0,0.00,10



=== FEATURES WITH NaN > 0 ===


,feature,dtype,missing_count,missing_pct,n_unique
27,TEAM,str,4364,78.24,30
28,INJURED_ON,str,4364,78.24,756
29,RETURNED,str,4364,78.24,736
30,DAYS_MISSED,float64,4364,78.24,64
31,INJURED_TYPE,str,4364,78.24,5
5,PLAYER_HEIGHT_INCHES,float64,6,0.11,22
6,PLAYER_WEIGHT,float64,6,0.11,124



=== REQUIRED FIELDS CLEANING ===
Rows dropped (NaN in ['PLAYER_HEIGHT_INCHES', 'PLAYER_WEIGHT']): 6
Rows remaining: 5,572

Saved cleaned data to: /home/ducvu0904/Downloads/xstk/injury+stat.csv
